# 00 · Dataset audit — CICIDS2017

This notebook runs **before** the schema, the ETL and the models. It answers four questions with evidence from your own files:

1. Which files are present, and how many rows does each contain?
2. Which file is not clean UTF-8, and which timestamp format does each file use?
3. What are the real column names, and which ones collide once cleaned?
4. What is the actual label distribution, including any label that fails to map?

It writes nothing to PostgreSQL. The only outputs are `reports/dataset_audit.json` and `reports/dataset_audit.md`, which later phases read.

**Phase 1 gate:** after running this you should be able to state the per-file row counts, the four data defects and the label families without opening a CSV again.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Jupyter starts in notebooks/, so lift the path one level to import config.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from config import settings

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 50)

print('project root   :', PROJECT_ROOT)
print('raw directory  :', settings.RAW_DIR)
print('reference rows :', f'{settings.EXPECTED_TOTAL_ROWS:,}')
print('pandas         :', pd.__version__)

## 1 · Inventory the raw capture files

The official `MachineLearningCSV.zip` contains 8 CSVs, split by capture day and by session. If the folder is empty, the cell prints the download steps instead of failing — that way the notebook is safe to open on a fresh clone.

In [ ]:
csv_files = sorted(settings.RAW_DIR.glob('*.csv'))
HAVE_DATA = len(csv_files) > 0

if HAVE_DATA:
    inventory = pd.DataFrame({
        'file': [path.name for path in csv_files],
        'size_mb': [round(path.stat().st_size / 1_000_000, 1) for path in csv_files],
    })
    print(f'{len(csv_files)} file(s) found, {inventory["size_mb"].sum():,.1f} MB total')
    print(f'reference release: {settings.EXPECTED_FILES} files, {settings.EXPECTED_TOTAL_ROWS:,} rows')
    print()
    print(inventory.to_string(index=False))
else:
    print(f'No CSV files found in {settings.RAW_DIR}')
    print('-' * 70)
    print('1. open https://www.unb.ca/cic/datasets/ids-2017.html and submit the short form')
    print('2. download MachineLearningCSV.zip (not the pcap archives)')
    print('3. unzip all 8 CSVs directly into data/raw/')
    print('4. re-run this notebook')

## 2 · Read each file with an encoding-safe reader

**Defect 1.** The Thursday `WebAttacks` file contains a `0xA0` byte inside a `Web Attack` label, so it is not valid UTF-8 and plain `pd.read_csv` raises `UnicodeDecodeError`. The reader below tries strict UTF-8 first, then `cp1252`, then `latin-1`, and reports which one each file needed. That record is the evidence for the claim "the dataset is not UTF-8 clean".

In [ ]:
def read_csv_any_encoding(path: Path):
    """Read one CICIDS2017 CSV, tolerating the non-UTF-8 byte. Returns (frame, encoding)."""
    for encoding in ('utf-8', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False), encoding
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f'Could not decode {path.name}')


frames = {}
file_encodings = {}

if HAVE_DATA:
    for path in csv_files:
        frame, encoding = read_csv_any_encoding(path)
        frames[path.stem] = frame
        file_encodings[path.name] = encoding
        print(f'{path.name:<56} {frame.shape[0]:>9,} rows  {frame.shape[1]:>3} cols  {encoding}')
    print()
    total_raw_rows = sum(frame.shape[0] for frame in frames.values())
    print(f'total rows across all files: {total_raw_rows:,}')
    print(f'difference from reference  : {total_raw_rows - settings.EXPECTED_TOTAL_ROWS:,}')
else:
    total_raw_rows = 0
    print('Skipping: no raw CSVs yet.')

## 3 · Column hygiene and timestamp parsing

**Defects 2 and 3.** Two separate problems that must be solved before a table can be designed:

- Column names arrive with leading spaces (`' Flow Packets/s'`) and one duplicated name (`Fwd Header Length` and `Fwd Header Length.1`). Cleaning them to snake_case without collisions is what makes a typed PostgreSQL table possible.
- The Thursday file uses a 12-hour clock with `AM`/`PM`; the Friday files use 24-hour `DD/MM/YYYY HH:MM`. The cell below *detects* the format per file rather than assuming one, and shows the match rate. A rate below 1.0 means the parse is wrong, not that the data is missing.

In [ ]:
TIMESTAMP_FORMATS = (
    '%d/%m/%Y %H:%M',
    '%d/%m/%Y %H:%M:%S',
    '%d/%m/%Y %I:%M:%S %p',
    '%m/%d/%Y %H:%M',
    '%m/%d/%Y %H:%M:%S',
    '%m/%d/%Y %I:%M:%S %p',
)


def clean_column_names(frame):
    """Lowercase, snake_case and de-duplicate column names.

    ' Flow Packets/s' -> 'flow_packets_s'
    'Fwd Header Length.1' -> 'fwd_header_length_1'
    """
    seen = {}
    renamed = []
    for raw in frame.columns:
        name = str(raw).strip().lower().replace(' ', '_')
        name = ''.join(ch if (ch.isalnum() or ch == '_') else '_' for ch in name)
        name = '_'.join(part for part in name.split('_') if part)
        if name in seen:
            seen[name] += 1
            name = f'{name}_{seen[name]}'
        else:
            seen[name] = 0
        renamed.append(name)
    out = frame.copy()
    out.columns = renamed
    return out


def detect_timestamp_format(series):
    """Return (format, match_rate) for the first format that parses every value."""
    best_format, best_rate = None, 0.0
    for fmt in TIMESTAMP_FORMATS:
        rate = float(pd.to_datetime(series, format=fmt, errors='coerce').notna().mean())
        if rate > best_rate:
            best_format, best_rate = fmt, rate
        if rate == 1.0:
            break
    return best_format, round(best_rate, 4)


cleaned = {}
column_rows = []
timestamp_rows = []

if HAVE_DATA:
    for name, raw_frame in frames.items():
        spaced = sum(1 for col in raw_frame.columns if str(col) != str(col).strip())
        frame = clean_column_names(raw_frame)
        cleaned[name] = frame

        if 'timestamp' in frame.columns:
            fmt, rate = detect_timestamp_format(frame['timestamp'])
            parsed = pd.to_datetime(frame['timestamp'], format=fmt, errors='coerce')
            first, last, unparsed = parsed.min(), parsed.max(), int(parsed.isna().sum())
        else:
            fmt, rate, first, last, unparsed = None, 0.0, None, None, frame.shape[0]

        column_rows.append({'file': name, 'columns': frame.shape[1], 'names_with_spacing': spaced})
        timestamp_rows.append({
            'file': name,
            'timestamp_format': fmt,
            'match_rate': rate,
            'unparsed': unparsed,
            'first': first,
            'last': last,
        })

    print('Column hygiene')
    print(pd.DataFrame(column_rows).to_string(index=False))
    print()
    print('Timestamp parsing')
    print(pd.DataFrame(timestamp_rows).to_string(index=False))
    print()
    first_file = next(iter(cleaned))
    print('sample cleaned names:', ', '.join(list(cleaned[first_file].columns)[:8]))
else:
    print('Skipping: no raw CSVs yet.')

## 4 · Label inventory

Confirms that the 14 attack labels plus `BENIGN` are present, that every raw spelling maps onto the canonical list in `config/settings.py`, and what the benign/attack split actually is. Any label printed as unexpected is a mapping bug to fix **before** the loader runs.

In [ ]:
label_counts = pd.Series(dtype='int64')
unexpected = []
total_rows = 0
attack_rows = 0

if HAVE_DATA:
    normalized = pd.concat(
        [frame['label'].map(settings.normalize_label) for frame in cleaned.values()],
        ignore_index=True,
    )
    label_counts = normalized.value_counts()
    unexpected = sorted(set(label_counts.index) - set(settings.CANONICAL_LABELS))

    total_rows = int(label_counts.sum())
    attack_rows = int(label_counts.drop(index=settings.BENIGN_LABEL, errors='ignore').sum())
    benign_rows = total_rows - attack_rows

    table = label_counts.rename_axis('label').reset_index(name='rows')
    table['family'] = table['label'].map(settings.LABEL_FAMILY)
    table['share_pct'] = (table['rows'] / total_rows * 100).round(3)
    print(table.to_string(index=False))
    print()
    print(f'total rows          : {total_rows:,}')
    print(f'benign rows         : {benign_rows:,}  ({benign_rows / total_rows:.2%})')
    print(f'attack rows         : {attack_rows:,}  ({attack_rows / total_rows:.2%})')
    print(f'reference attack rows: {settings.REFERENCE_ATTACK_ROWS:,}')
    print(f'difference           : {attack_rows - settings.REFERENCE_ATTACK_ROWS:,}')
    print()
    if unexpected:
        print('LABELS OUTSIDE CANONICAL_LABELS - fix settings.normalize_label before loading:')
        print(unexpected)
    else:
        print('every label maps onto CANONICAL_LABELS')
else:
    print('Skipping: no raw CSVs yet.')

## 5 · Missing values, infinities, duplicates and memory

**Defect 4.** `Flow Bytes/s` and `Flow Packets/s` are divisions by `Flow Duration`, so a zero-duration flow produces `inf`. Counting them here decides the cleaning rule in Phase 2: `inf` becomes null and is reported, never silently turned into zero.

The duplicate count is what the loader's `ON CONFLICT DO NOTHING` strategy and the `dedupe` step must actually remove. Memory tells us whether the 200,000-row batch size in `.env` is sensible for this machine.

In [ ]:
quality_rows = []

if HAVE_DATA:
    for name, frame in cleaned.items():
        numeric = frame.select_dtypes(include=[np.number])
        inf_cells = 0
        if numeric.shape[1]:
            values = numeric.to_numpy(dtype='float64', na_value=np.nan)
            inf_cells = int(np.isinf(values).sum())

        quality_rows.append({
            'file': name,
            'rows': int(frame.shape[0]),
            'null_cells': int(frame.isna().sum().sum()),
            'columns_with_nulls': int((frame.isna().sum() > 0).sum()),
            'inf_cells': inf_cells,
            'duplicate_rows': int(frame.duplicated().sum()),
            'memory_mb': round(frame.memory_usage(deep=True).sum() / 1_000_000, 1),
        })

    quality = pd.DataFrame(quality_rows)
    print(quality.to_string(index=False))
    print()
    print(f'total duplicate rows: {quality["duplicate_rows"].sum():,}')
    print(f'total inf cells      : {quality["inf_cells"].sum():,}')
    print(f'peak in-memory frame : {quality["memory_mb"].max():,.0f} MB')
else:
    print('Skipping: no raw CSVs yet.')

## 6 · Write the audit artefacts, then check the gate

`reports/dataset_audit.json` is the machine-readable contract that later phases and the tests read. `reports/dataset_audit.md` is the human summary you can paste into the project report.

Do not start Phase 2 until every line of the gate prints `[x]`.

In [ ]:
def mark(condition):
    return 'x' if condition else ' '


if not HAVE_DATA:
    print('Skipping: no raw CSVs yet.')
else:
    generated_at = pd.Timestamp.now(tz='UTC').isoformat()
    encodings_by_stem = {Path(name).stem: enc for name, enc in file_encodings.items()}

    per_file = {}
    for name, frame in cleaned.items():
        encodings_entry = encodings_by_stem.get(name)
        per_file[name] = {
            'encoding': encodings_entry,
            'rows': int(frame.shape[0]),
            'columns': int(frame.shape[1]),
            'memory_mb': round(frame.memory_usage(deep=True).sum() / 1_000_000, 1),
        }

    audit = {
        'dataset': 'CICIDS2017 - official MachineLearningCSV release',
        'generated_at': generated_at,
        'totals': {
            'rows': total_rows,
            'expected_rows': settings.EXPECTED_TOTAL_ROWS,
            'rows_match_reference': total_rows == settings.EXPECTED_TOTAL_ROWS,
            'attack_rows': attack_rows,
            'reference_attack_rows': settings.REFERENCE_ATTACK_ROWS,
        },
        'timestamp_formats': {row['file']: row['timestamp_format'] for row in timestamp_rows},
        'labels': {str(label): int(count) for label, count in label_counts.items()},
        'unexpected_labels': unexpected,
        'per_file': per_file,
        'quality': quality_rows,
    }

    json_path = settings.REPORTS_DIR / 'dataset_audit.json'
    json_path.write_text(json.dumps(audit, indent=2, default=str), encoding='utf-8')

    share_text = f'{attack_rows / total_rows:.2%}' if total_rows else 'n/a'
    md_lines = [
        '# Dataset audit - CICIDS2017',
        '',
        'Generated ' + generated_at,
        '',
        '## Totals',
        '',
        '| Metric | Value |',
        '| --- | --- |',
        '| Files read | ' + str(len(cleaned)) + ' |',
        '| Rows | ' + f'{total_rows:,}' + ' |',
        '| Reference rows | ' + f'{settings.EXPECTED_TOTAL_ROWS:,}' + ' |',
        '| Attack rows | ' + f'{attack_rows:,}' + ' |',
        '| Attack share | ' + share_text + ' |',
        '',
        '## Labels',
        '',
        '| Label | Family | Rows |',
        '| --- | --- | --- |',
    ]
    for label, count in label_counts.items():
        family = settings.LABEL_FAMILY.get(label) or 'benign'
        md_lines.append('| ' + label + ' | ' + str(family) + ' | ' + f'{int(count):,}' + ' |')

    md_lines += [
        '',
        '## Timestamp formats',
        '',
        '| File | Format | Match rate |',
        '| --- | --- | --- |',
    ]
    for row in timestamp_rows:
        md_lines.append('| ' + row['file'] + ' | ' + str(row['timestamp_format']) + ' | ' + str(row['match_rate']) + ' |')

    md_lines += ['', '## Quality', '', '| File | Duplicates | Inf cells | Null cells | MB |', '| --- | --- | --- | --- | --- |']
    for row in quality_rows:
        md_lines.append(
            '| ' + row['file'] + ' | ' + f'{row["duplicate_rows"]:,}'
            + ' | ' + f'{row["inf_cells"]:,}'
            + ' | ' + f'{row["null_cells"]:,}'
            + ' | ' + str(row['memory_mb']) + ' |'
        )

    md_path = settings.REPORTS_DIR / 'dataset_audit.md'
    md_path.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')

    print('written:', json_path)
    print('written:', md_path)
    print()
    print('Phase 1 gate')
    print('-' * 70)
    print(f'[{mark(len(cleaned) == settings.EXPECTED_FILES)}] all {settings.EXPECTED_FILES} capture files loaded')
    print(f'[{mark(total_rows == settings.EXPECTED_TOTAL_ROWS)}] row count matches the reference total')
    print(f'[{mark(not unexpected)}] every label maps onto CANONICAL_LABELS')
    print(f'[{mark(any(enc != "utf-8" for enc in file_encodings.values()))}] non-UTF-8 file detected and read')
    print(f'[{mark(len(set(row["timestamp_format"] for row in timestamp_rows)) > 1)}] mixed timestamp formats detected')